In [ ]:
import sys
import os

ROOT_DIR = '../../..'

from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import albumentations as Albu
import cv2
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, recall_score, precision_score
from tqdm import tqdm
from skimage.color import rgb2hed, hed2rgb # Adicionado para manipulação HED

sys.path.append("../../..")
from utils.dataset import PandasDataset
from utils.models import EfficientNetApi

## Configuration

In [ ]:
seed = 42
batch_size = 3
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
dropout_rate = 0.6
patience = 7

focal_alpha = 0.25
focal_gamma = 0.2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

data_dir = '../../../..'
images_dir = os.path.join(data_dir, 'tiles')

os.makedirs('../logs', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Atualizado para refletir o uso do HED
model_path = 'models/b0-entropy-ordinal-focal-hed.pth'
log_path = 'logs/b0-entropy-ordinal-focal-hed.txt'

Using device: cuda


In [ ]:
from albumentations.core.transforms_interface import ImageOnlyTransform

class HEDEnhancement(ImageOnlyTransform):
    """
    Processamento no espaço HED (Hematoxilina, Eosina, DAB).
    Pipeline: RGB → HED → Stain Augmentation (opcional) → Zera DAB → CLAHE(H) (opcional) → RGB
    """
    def __init__(self, augment_stain=False, alpha_range=(0.85, 1.15), beta_range=(0.85, 1.15),
                 apply_clahe_h=True, clip_limit=2.0, tile_grid_size=(8, 8), p=1.0):
        super().__init__(p=p)
        self.augment_stain = augment_stain
        self.alpha_range = alpha_range
        self.beta_range = beta_range
        self.apply_clahe_h = apply_clahe_h
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

        if self.apply_clahe_h:
            self._clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def apply(self, image, **kwargs):
        # 1. Converter para HED
        hed = rgb2hed(image)

        # 2. Stain Augmentation (varia as concentrações de corante sinteticamente)
        if self.augment_stain:
            alpha = np.random.uniform(*self.alpha_range)
            beta  = np.random.uniform(*self.beta_range)
            hed[:, :, 0] *= alpha # Varia Hematoxilina
            hed[:, :, 1] *= beta  # Varia Eosina

        # 3. Remover ruído/artefatos zerando o canal DAB (marrom)
        hed[:, :, 2] = 0.0

        # 4. Aplicar CLAHE apenas no canal H (núcleos)
        if self.apply_clahe_h:
            h_channel = hed[:, :, 0]
            h_min, h_max = np.min(h_channel), np.max(h_channel)

            # CLAHE do OpenCV exige uint8, precisamos mapear de float para 0-255
            if h_max > h_min:
                h_norm = ((h_channel - h_min) / (h_max - h_min) * 255).astype(np.uint8)
                h_clahe = self._clahe.apply(h_norm)
                # Retornar para a escala original
                hed[:, :, 0] = (h_clahe.astype(np.float32) / 255.0) * (h_max - h_min) + h_min

        # 5. Voltar para RGB
        rgb_recon = hed2rgb(hed)
        rgb_recon = np.clip(rgb_recon * 255.0, 0, 255).astype(np.uint8)

        return rgb_recon

    def get_transform_init_args_dict(self):
        return {"augment_stain": self.augment_stain, "alpha_range": self.alpha_range,
                "beta_range": self.beta_range, "apply_clahe_h": self.apply_clahe_h,
                "clip_limit": self.clip_limit, "tile_grid_size": self.tile_grid_size}

## Loss: Ordinal Loss + Focal Loss (γ=0.2)

Two separate terms are combined:

1. **`OrdinalLoss`** — plain BCE applied to ordinal-encoded targets.  Provides the
   base ordinal regression signal that enforces the grade ordering.

2. **`OrdinalFocalLoss`** — focal-weighted BCE on the same ordinal targets with γ=0.2.
   Mildly focuses learning on hard threshold boundaries without discarding easy ones.

```
total_loss = ordinal_loss + focal_loss
           = BCE(sigmoid(logits), ordinal_targets)
           + alpha * (1 - p_t)^0.2 * BCE(sigmoid(logits), ordinal_targets)
```

In [4]:
class FocalOrdinalRegressionLoss(nn.Module):
    def __init__(self, num_classes=5, alpha=1.0, beta=1.0, gamma=2.0):
        """
        num_classes : número de thresholds ordinais (ISUP: 5)
        alpha       : peso da focal loss
        beta        : peso da ordinal loss
        gamma       : fator de modulação focal
        """
        super().__init__()
        self.alpha       = alpha
        self.beta        = beta
        self.gamma       = gamma
        self.num_classes = num_classes
        self.C2          = num_classes ** 2

    def focal_loss(self, logits, targets):
        probs  = torch.sigmoid(logits)
        bce    = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t    = probs * targets + (1 - probs) * (1 - targets)
        return (((1 - p_t) ** self.gamma) * bce).mean()

    def ordinal_loss(self, logits, targets):
        probs = torch.sigmoid(logits)                        # (B, C)

        # ── Expected value via probabilidade acumulada ────────────────────────
        # P(Y > k) = prob do threshold k estar ativo
        # E[Y] = sum_k P(Y > k)  →  isso é matematicamente correto para
        # variáveis ordinais com encoding cumulativo
        expected_class = probs.sum(dim=1)                    # (B,)
        targets_class  = targets.sum(dim=1)                  # (B,)

        # ── Penalização quadrática normalizada ────────────────────────────────
        # Divide por C² para manter em [0, 1] independente do num_classes
        # Garante que alpha e beta ficam na mesma escala que a focal
        squared_error  = (expected_class - targets_class) ** 2  # (B,)
        return squared_error.mean() / self.C2

    def forward(self, logits, targets):
        targets = targets.to(logits.device)
        f_loss  = self.focal_loss(logits, targets)
        o_loss  = self.ordinal_loss(logits, targets)
        return self.alpha * f_loss + self.beta * o_loss


loss_fn = FocalOrdinalRegressionLoss(
    num_classes=5,
    alpha=1.0,
    gamma=2.0,
    beta=1.0,
)


def decode_ordinal_predictions(logits):
    return (torch.sigmoid(logits) > 0.5).sum(dim=1)


loss_function = FocalOrdinalRegressionLoss(alpha=focal_alpha, gamma=focal_gamma)
print(f"Loss: OrdinalLoss + OrdinalFocalLoss")
print(f"  alpha: {focal_alpha}")
print(f"  gamma: {focal_gamma}")

Loss: OrdinalLoss + OrdinalFocalLoss
  alpha: 0.25
  gamma: 0.2


## Load Data with Entropy Filtering

In [5]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
print(f"Original records: {len(df_train_)}")

df_entropy = pd.read_csv(f"{ROOT_DIR}/data/entropy.csv")
print(f"High-entropy samples to remove: {len(df_entropy)}")

df_entropy_sorted = df_entropy.sort_values(by='difficulty_score', ascending=False).reset_index(drop=True)
n_remove          = int(len(df_entropy) * 0.2)
df_entropy_top    = df_entropy_sorted.head(n_remove)

df_train_ = df_train_[~df_train_['image_id'].isin(df_entropy_top['image_id'])].reset_index(drop=True)
print(f"Filtered records: {len(df_train_)}")

df_train_.columns = df_train_.columns.str.strip()

train_indexes = np.where(df_train_['fold'] != 3)[0]
valid_indexes = np.where(df_train_['fold'] == 3)[0]

df_train = df_train_.loc[train_indexes].reset_index(drop=True)
df_val   = df_train_.loc[valid_indexes].reset_index(drop=True)
df_test  = pd.read_csv(f"{ROOT_DIR}/data/test.csv")


def remove_nonexistent_images(df, images_dir):
    paths     = df['image_id'].apply(lambda x: os.path.join(images_dir, f"{x}.png"))
    existing  = [os.path.isfile(p) for p in paths]
    return df[existing]


df_train = remove_nonexistent_images(df_train, images_dir)
df_val   = remove_nonexistent_images(df_val,   images_dir)
df_test  = remove_nonexistent_images(df_test,  images_dir)

print(f"\nTrain:      {len(df_train)} samples")
print(f"Validation: {len(df_val)} samples")
print(f"Test:       {len(df_test)} samples")
print(f"\nTrain class distribution:")
print(df_train['isup_grade'].value_counts().sort_index())

Original records: 9024
High-entropy samples to remove: 903
Filtered records: 8844

Train:      7073 samples
Validation: 1767 samples
Test:       1590 samples

Train class distribution:
isup_grade
0    1953
1    1802
2     899
3     826
4     832
5     761
Name: count, dtype: int64


## Data Augmentation with CLAHE-YUV

`CLAHEOnYChannel` enhances luminance contrast (p=1.0 — applied to every sample) alongside the standard geometric augmentations. The colour channels are unmodified.

In [ ]:
train_transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    HEDEnhancement(augment_stain=True, apply_clahe_h=True, clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
])

# Na validação/teste, usamos CLAHE nos núcleos e removemos DAB, mas NÃO usamos variância sintética (Stain Aug)
val_transforms = Albu.Compose([
    HEDEnhancement(augment_stain=False, apply_clahe_h=True, clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
])

/tmp/ipykernel_312592/2850931843.py:12: UserWarning: Argument(s) 'always_apply' are not valid for transform BasicTransform
  super().__init__(always_apply=always_apply, p=p)


## Create Datasets and DataLoaders

In [7]:
from torch.utils.data import SequentialSampler
train_dataset = PandasDataset(images_dir, df_train, transforms=train_transforms, format="png")
valid_dataset = PandasDataset(images_dir, df_val,   transforms=val_transforms,   format="png")
test_dataset  = PandasDataset(images_dir, df_test,  transforms=val_transforms,   format="png")

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    num_workers=num_workers,
    shuffle=True,
    pin_memory=True, persistent_workers=True)
valid_loader = DataLoader(
    valid_dataset, 
    batch_size=batch_size, 
    num_workers=num_workers,
    shuffle=False,
    pin_memory=True, 
    persistent_workers=True
)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, num_workers=num_workers,
                          shuffle=False, pin_memory=True, persistent_workers=True)

print(f"Train batches:      {len(train_loader)}")
print(f"Validation batches: {len(valid_loader)}")
print(f"Test batches:       {len(test_loader)}")

Train batches:      2358
Validation batches: 589
Test batches:       530


## Model Setup

In [8]:
load_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=dropout_rate)
model = model.to(device)

print(f"Model loaded on {device}")
print(f"Output dimensions: {output_classes} (ordinal thresholds)")
print(f"Dropout rate: {dropout_rate}")

Model loaded on cuda
Output dimensions: 5 (ordinal thresholds)
Dropout rate: 0.6


## Optimizer and Scheduler

In [ ]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

optimizer = optim.Adam(model.parameters(), lr=init_lr)

# Warmup: cresce de init_lr/warmup_factor até init_lr
scheduler_warmup = LinearLR(
    optimizer,
    start_factor=1 / warmup_factor,
    end_factor=1.0,
    total_iters=warmup_epochs,
)

# Cosine annealing após o warmup
scheduler_cosine = CosineAnnealingLR(
    optimizer,
    T_max=n_epochs - warmup_epochs,
    eta_min=1e-6,  # lr mínimo ao final, evita chegar a 0
)

# Combina os dois sequencialmente
scheduler = SequentialLR(
    optimizer,
    schedulers=[scheduler_warmup, scheduler_cosine],
    milestones=[warmup_epochs],
)

Optimizer and scheduler configured


## Training and Validation Functions

In [16]:
def training_step(model, dataloader, optimizer, device, loss_fn, scaler):
    model.train()
    train_loss = []
    running_loss = 0.0
    bar_progress = tqdm(dataloader, desc="Training")

    for step, (batch_data, batch_targets, _) in enumerate(bar_progress):
        batch_data    = batch_data.to(device, non_blocking=True)
        batch_targets = batch_targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        
        with autocast(device_type='cuda'):
            logits = model(batch_data)
            loss = loss_fn(logits, batch_targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_value = loss.item()
        train_loss.append(loss_value)

        running_loss += loss_value
        smooth_loss = running_loss / (step + 1)

        bar_progress.set_postfix({
            'loss': f'{loss_value:.5f}',
            'smooth': f'{smooth_loss:.5f}'
        })

    return train_loss


def validation_step(model, dataloader, device, loss_fn):
    model.eval()

    validation_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad(), autocast(device_type='cuda'):
        for batch_data, batch_targets, _ in tqdm(dataloader, desc="Validation", leave=False):
            batch_data = batch_data.to(device, non_blocking=True)
            batch_targets = batch_targets.to(device, non_blocking=True)

            logits = model(batch_data)
            loss = loss_fn(logits, batch_targets)

            predictions = decode_ordinal_predictions(logits)
            targets_class = batch_targets.sum(dim=1).long()

            all_preds.append(predictions.cpu())
            all_targets.append(targets_class.cpu())

            validation_loss += loss.item()

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    n_batches = len(dataloader)

    return {
        'val_loss': validation_loss / n_batches,
        'val_acc': accuracy_score(all_targets, all_preds),
        'val_kappa': cohen_kappa_score(all_targets, all_preds, weights='quadratic'),
        'val_f1': f1_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_recall': recall_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_precision': precision_score(all_targets, all_preds, average='macro', zero_division=0),
    }

## Training Loop

In [ ]:
best_kappa                = 0.0
best_epoch                = 0
epochs_without_improvement = 0

history = {
    'train_loss': [], 'val_loss': [], 'val_acc': [],
    'val_kappa': [], 'val_f1': [], 'val_recall': [], 'val_precision': []
}

print("\nStarting training — OrdinalLoss + FocalLoss(γ=0.2) + CLAHE-YUV\n")
print("="*80)

from torch.amp import autocast, GradScaler

scaler = GradScaler()

for epoch in range(1, n_epochs + 1):
    print(f"\nEpoch {epoch}/{n_epochs}")
    print("-" * 80)

    train_loss = training_step(model, train_loader, optimizer, device, loss_function, scaler)
    metrics    = validation_step(model, valid_loader, device, loss_function)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(np.mean(train_loss))
    history['val_loss'].append(metrics['val_loss'])
    history['val_acc'].append(metrics['val_acc'])
    history['val_kappa'].append(metrics['val_kappa'])
    history['val_f1'].append(metrics['val_f1'])
    history['val_recall'].append(metrics['val_recall'])
    history['val_precision'].append(metrics['val_precision'])

    print(f"\n  Train Loss:    {history['train_loss'][-1]:.5f}")
    print(f"  Val Loss:      {metrics['val_loss']:.5f}")
    print(f"  Val Accuracy:  {metrics['val_acc']*100:.2f}%")
    print(f"  Val Kappa:     {metrics['val_kappa']:.4f}")
    print(f"  Val F1:        {metrics['val_f1']:.4f}")
    print(f"  Learning Rate: {current_lr:.7f}")

    log_line = (
        f"epoch: {epoch} | lr: {current_lr:.7f} | "
        f"train_loss: {history['train_loss'][-1]:.5f} | "
        f"val_loss: {metrics['val_loss']:.5f} | "
        f"val_acc: {metrics['val_acc']:.4f} | "
        f"val_kappa: {metrics['val_kappa']:.4f}\n"
    )
    with open(log_path, 'a') as f:
        f.write(log_line)

    if metrics['val_kappa'] > best_kappa:
        best_kappa                 = metrics['val_kappa']
        best_epoch                 = epoch
        epochs_without_improvement = 0
        torch.save(model.state_dict(), model_path)
        print(f"\n  Best model saved! Kappa: {best_kappa:.4f}")
    else:
        epochs_without_improvement += 1
        print(f"\n  No improvement for {epochs_without_improvement} epoch(s)")

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping at epoch {epoch} — best epoch: {best_epoch}, kappa: {best_kappa:.4f}")
        break

print("\n" + "="*80)
print(f"Training complete. Best kappa: {best_kappa:.4f} at epoch {best_epoch}")
print("="*80)


Starting training — OrdinalLoss + FocalLoss(γ=0.2) + CLAHE-YUV


Epoch 1/50
--------------------------------------------------------------------------------


Training: 100%|██████████| 2358/2358 [08:20<00:00,  4.71it/s, loss=0.03233, smooth=0.11505]



  Train Loss:    0.11505
  Val Loss:      0.09403
  Val Accuracy:  61.40%
  Val Kappa:     0.8161
  Val F1:        0.5269
  Learning Rate: 0.0003000

  Best model saved! Kappa: 0.8161

Epoch 2/50
--------------------------------------------------------------------------------


Training: 100%|██████████| 2358/2358 [08:26<00:00,  4.66it/s, loss=0.04125, smooth=0.11361]
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1087: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)



  Train Loss:    0.11361
  Val Loss:      0.09903
  Val Accuracy:  58.06%
  Val Kappa:     0.8118
  Val F1:        0.5048
  Learning Rate: 0.0003003

  No improvement for 1 epoch(s)

Epoch 3/50
--------------------------------------------------------------------------------


Training: 100%|██████████| 2358/2358 [08:32<00:00,  4.60it/s, loss=0.09781, smooth=0.10894]



  Train Loss:    0.10894
  Val Loss:      0.09693
  Val Accuracy:  59.82%
  Val Kappa:     0.8103
  Val F1:        0.5047
  Learning Rate: 0.0003000

  No improvement for 2 epoch(s)

Epoch 4/50
--------------------------------------------------------------------------------


Training:  30%|██▉       | 698/2358 [02:31<05:53,  4.70it/s, loss=0.07458, smooth=0.10771]

## Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(history['train_loss'], label='Train Loss')
axes[0, 0].plot(history['val_loss'],   label='Val Loss')
axes[0, 0].set_title('Loss (Ordinal + Focal γ=0.2 + CLAHE-YUV)')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend(); axes[0, 0].grid(True)

axes[0, 1].plot(history['val_acc'], label='Val Accuracy', color='green')
axes[0, 1].set_title('Validation Accuracy')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend(); axes[0, 1].grid(True)

axes[1, 0].plot(history['val_kappa'], label='Val Kappa', color='orange')
axes[1, 0].set_title('Validation Kappa')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('QW Kappa')
axes[1, 0].legend(); axes[1, 0].grid(True)

axes[1, 1].plot(history['val_f1'], label='Val F1', color='red')
axes[1, 1].set_title('Validation F1 Score')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Macro F1')
axes[1, 1].legend(); axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('logs/b0-entropy-ordinal-focal-yhuv-clahe-training.png', dpi=300, bbox_inches='tight')
plt.show()

## Evaluation on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.load_state_dict(torch.load(model_path, weights_only=True))
model.eval()

test_preds   = []
test_targets = []

with torch.no_grad():
    for batch_data, batch_targets, _ in tqdm(test_loader, desc="Testing"):
        batch_data = batch_data.to(device)
        logits     = model(batch_data)
        test_preds.append(decode_ordinal_predictions(logits).cpu())
        test_targets.append(batch_targets.sum(dim=1).long())

test_preds   = torch.cat(test_preds).numpy()
test_targets = torch.cat(test_targets).numpy()

# ── Point estimates ────────────────────────────────────────────────────────────
test_accuracy = accuracy_score(test_targets, test_preds)
test_kappa    = cohen_kappa_score(test_targets, test_preds, weights='quadratic')
test_f1       = f1_score(test_targets, test_preds, average='macro', zero_division=0)

# ── Bootstrap CI (1 000 resamples) ────────────────────────────────────────────
N_BOOTSTRAP = 1000
rng = np.random.default_rng(seed=42)
n   = len(test_targets)
boot_acc   = np.empty(N_BOOTSTRAP)
boot_kappa = np.empty(N_BOOTSTRAP)
boot_f1    = np.empty(N_BOOTSTRAP)

for i in tqdm(range(N_BOOTSTRAP), desc="Bootstrap"):
    idx           = rng.integers(0, n, size=n)
    boot_acc[i]   = accuracy_score(test_targets[idx], test_preds[idx])
    boot_kappa[i] = cohen_kappa_score(test_targets[idx], test_preds[idx], weights='quadratic')
    boot_f1[i]    = f1_score(test_targets[idx], test_preds[idx], average='macro', zero_division=0)

def boot_stats(arr):
    return arr.std(ddof=1), np.percentile(arr, 2.5), np.percentile(arr, 97.5)

acc_std,   acc_lo,   acc_hi   = boot_stats(boot_acc)
kappa_std, kappa_lo, kappa_hi = boot_stats(boot_kappa)
f1_std,    f1_lo,    f1_hi    = boot_stats(boot_f1)

print("\n" + "="*80)
print("TEST SET RESULTS — Ordinal + Focal (γ=0.2) + CLAHE-YUV")
print("="*80)
print(f"Accuracy : {test_accuracy*100:.2f}% ± {acc_std*100:.2f}%  [{acc_lo*100:.2f}% – {acc_hi*100:.2f}%]")
print(f"QW Kappa : {test_kappa:.4f} ± {kappa_std:.4f}  [{kappa_lo:.4f} – {kappa_hi:.4f}]")
print(f"Macro F1 : {test_f1:.4f} ± {f1_std:.4f}  [{f1_lo:.4f} – {f1_hi:.4f}]")
print("="*80)
print(classification_report(test_targets, test_preds,
      target_names=[f'ISUP {i}' for i in range(6)], digits=4, zero_division=0))

cm = confusion_matrix(test_targets, test_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'ISUP {i}' for i in range(6)],
            yticklabels=[f'ISUP {i}' for i in range(6)])
plt.title('Confusion Matrix — Ordinal + Focal (γ=0.2) + CLAHE-YUV', fontweight='bold')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('logs/b0-entropy-ordinal-focal-yhuv-clahe-confusion-matrix.png', dpi=300, bbox_inches='tight')
plt.show()

with open('logs/b0-entropy-ordinal-focal-yhuv-clahe-test-results.txt', 'w') as f:
    f.write("EfficientNet-B0 — Entropy + OrdinalLoss + FocalLoss(γ=0.2) + CLAHE-YUV\n")
    f.write("="*80 + "\n\n")
    f.write(f"Bootstrap resamples: {N_BOOTSTRAP}\n\n")
    f.write(f"Accuracy : {test_accuracy*100:.2f}% ± {acc_std*100:.2f}%  [{acc_lo*100:.2f}% – {acc_hi*100:.2f}%]\n")
    f.write(f"QW Kappa : {test_kappa:.4f} ± {kappa_std:.4f}  [{kappa_lo:.4f} – {kappa_hi:.4f}]\n")
    f.write(f"Macro F1 : {test_f1:.4f} ± {f1_std:.4f}  [{f1_lo:.4f} – {f1_hi:.4f}]\n\n")
    f.write(classification_report(test_targets, test_preds,
            target_names=[f'ISUP {i}' for i in range(6)], digits=4, zero_division=0))
    f.write("\nConfusion Matrix:\n" + str(cm))

print("Saved: logs/b0-entropy-ordinal-focal-yhuv-clahe-test-results.txt")

In [ ]:
cm_norm  = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
classes  = [f'ISUP {i}' for i in range(6)]

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Normalized Confusion Matrix — Ordinal + Focal (γ=0.2) + CLAHE-YUV')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('logs/b0-entropy-ordinal-focal-yhuv-clahe-confusion-matrix-normalized.png',
            dpi=300, bbox_inches='tight')
plt.show()

## Summary

| Component | Detail |
|---|---|
| Entropy filtering | Top 20% hardest samples removed |
| **OrdinalLoss** | Plain BCE on ordinal-encoded targets |
| **OrdinalFocalLoss** | Focal-weighted BCE, α=0.25, γ=0.2 |
| **Combined loss** | `total = OrdinalLoss + OrdinalFocalLoss` |
| **CLAHE-YUV preprocessing** | CLAHE (clip_limit=2.0, tile=8×8) on Y channel of YUV — colour (U/V) preserved |
| **Colour space pipeline** | RGB → YUV → CLAHE(Y) → YUV → RGB |